In [1]:
import sys, os
import pickle
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
from IPython.display import Image
import pyro
import torch

data_type = 'float32'
os.environ["THEANO_FLAGS"] = f"device=cuda,floatX={data_type},force_device=True,dnn.enabled=False"

In [2]:
#Reload Variational Means
#Can Restart the Kernel for Memory 
import torch

CTRL_means = torch.load("CTRL_variational_means.pt", map_location="cpu")
ADCAA_means = torch.load("ADCAA_variational_means.pt", map_location="cpu")

In [3]:
state_ADCAA = torch.load("ADCAA_param_store.pt")
# Populate Pyro param store
for name, value in state_ADCAA.items():
    pyro.param(name, torch.tensor(value) if not torch.is_tensor(value) else value)

# Extract all '_locs' parameters
ADCAA_means = {
    name.replace(".locs", ""): param.detach().cpu()
    for name, param in pyro.get_param_store().items()
    if ".locs" in name
}
spot_factors = ADCAA_means['AutoNormal.spot_factors']
print(spot_factors.shape)  # check dimensions

torch.Size([145, 46])


In [4]:
# Load Control param store
state_CTRL = torch.load("CTRL_param_store.pt", map_location="cpu")

pyro.clear_param_store()
for name, value in state_CTRL.items():
    pyro.param(name, value)

# Extract posterior means
CTRL_means = {
    name.replace(".locs", ""): param.detach().cpu()
    for name, param in pyro.get_param_store().items()
    if ".locs" in name
}

spot_factors_CTRL = CTRL_means["AutoNormal.spot_factors"]
print(spot_factors_CTRL.shape)

torch.Size([45, 46])


In [5]:
#Normalization of spot factors
#relative cell-type abundance per GeoMx Spot
#Normalize separately
spot_props_ADCAA = spot_factors / spot_factors.sum(dim=1, keepdim=True)
spot_props_CTRL  = spot_factors_CTRL / spot_factors_CTRL.sum(dim=1, keepdim=True)

In [6]:
# --- CONFIG: change these to your paths ---
data_path = "/N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/"   # <-- base folder
spatial_h5ad = os.path.join(data_path, "CAA-AD_AnnData.h5ad")            # <-- GeoMx WTA spatial AnnData
results_root = os.path.join(data_path, "SpaceJam_results")               # where we'll save outputs
# Path where your inferred signatures are stored (according to "Regression.ipynb")
sig_dir = os.path.join(data_path, "Regression-model")
sig_files = {
    "AD+CAA": os.path.join(sig_dir, "AD+CAA_inferred_signatures.csv"),   # from previous notebook
    "Control": os.path.join(sig_dir, "Control_inferred_signatures.csv")
}
nucleus_count = os.path.join(data_path, "nuc_count.csv")            # <-- GeoMx WTA spatial nuclei count
# --- END CONFIG ---
os.makedirs(results_root, exist_ok=True)

In [7]:
#Load Nanostring WTA data 
adata_wta = sc.read_h5ad(spatial_h5ad)
print("Loaded spatial AnnData:", adata_wta)

# Ensure layers/raw exist
if "counts" not in adata_wta.layers:
    # If X is raw counts, copy it to layers['counts']
    print("No layers['counts'] found — copying .X to layers['counts']")
    adata_wta.layers["counts"] = adata_wta.X.copy()
    
#Split WTA object by "Disease Status"
# Check unique values first
print(adata_wta.obs["disease_status"].unique())

# Split
adata_wta_ADCAA = adata_wta[adata_wta.obs["disease_status"] == "AD-CAA"].copy()
adata_wta_Control = adata_wta[adata_wta.obs["disease_status"] == "Control"].copy()

# Confirm sizes
print(adata_wta_ADCAA.shape)
print(adata_wta_Control.shape)

Loaded spatial AnnData: AnnData object with n_obs × n_vars = 190 × 18676
    obs: 'Scan Name', 'ROI', 'Segment (Name/ Label)', 'Segment Tags', 'Custom Segment Name', 'LOT_Human_NGS_Whole_Transcriptome_Atlas_RNA_1_0', 'ROI_ID', 'Scan_ID', 'disease_status', 'pathology', 'region'
    var: 'TargetName', 'ProbeID', 'Negative'
    obsm: 'negProbes'
No layers['counts'] found — copying .X to layers['counts']
['AD-CAA', 'Control']
Categories (2, object): ['AD-CAA', 'Control']
(145, 18676)
(45, 18676)


/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [8]:
#Attach Metadata separately 
#ADCAA
spot_props_ADCAA_df = pd.DataFrame(
    spot_props_ADCAA.numpy(),
    columns=[f"Factor_{i+1}" for i in range(spot_props_ADCAA.shape[1])]
)

spot_props_ADCAA_df["spot_id"] = adata_wta_ADCAA.obs_names.values
spot_props_ADCAA_df["condition"] = "AD-CAA"
spot_props_ADCAA_df["Scan_ID"] = adata_wta_ADCAA.obs["Scan_ID"].values
#Control
spot_props_CTRL_df = pd.DataFrame(
    spot_props_CTRL.numpy(),
    columns=[f"Factor_{i+1}" for i in range(spot_props_CTRL.shape[1])]
)

spot_props_CTRL_df["spot_id"] = adata_wta_Control.obs_names.values
spot_props_CTRL_df["condition"] = "Control"
spot_props_CTRL_df["Scan_ID"] = adata_wta_Control.obs["Scan_ID"].values

In [9]:
#Concatenation 
spot_props_all = pd.concat(
    [spot_props_ADCAA_df, spot_props_CTRL_df],
    axis=0,
    ignore_index=True
)

In [10]:
assert spot_props_ADCAA_df.shape[0] == adata_wta_ADCAA.n_obs
assert spot_props_CTRL_df.shape[0] == adata_wta_Control.n_obs
assert spot_props_ADCAA_df.shape[1] == spot_props_CTRL_df.shape[1]
#Must not throw errors

In [11]:
#Load reference signatures (snRNAseq)
#Path to Regression results
rresults = '/N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/Regression-model/'
X_ref_ADCAA = pd.read_csv(
    f"{rresults}/AD+CAA_inferred_signatures.csv",
    index_col=0
)

X_ref_Control = pd.read_csv(
    f"{rresults}/Control_inferred_signatures.csv",
    index_col=0
)
#Intersection of Genes
def align_ref_and_spatial(X_ref, adata_wta):
    common = adata_wta.var_names.intersection(X_ref.index)
    print(f"Shared genes: {len(common)}")

    adata_sub = adata_wta[:, common].copy()
    X_ref_sub = X_ref.loc[common, :]

    return X_ref_sub, adata_sub

X_ref_ADCAA_sub, adata_wta_ADCAA_sub = align_ref_and_spatial(
    X_ref_ADCAA, adata_wta_ADCAA
)

X_ref_Control_sub, adata_wta_Control_sub = align_ref_and_spatial(
    X_ref_Control, adata_wta_Control
)

Shared genes: 14620
Shared genes: 14620


In [12]:
#Raw and relative abundance
# Absolute abundance
spot_abs_ADCAA = spot_factors
spot_abs_CTRL  = spot_factors_CTRL

# Relative abundance
spot_rel_ADCAA = spot_abs_ADCAA / spot_abs_ADCAA.sum(dim=1, keepdim=True)
spot_rel_CTRL  = spot_abs_CTRL  / spot_abs_CTRL.sum(dim=1, keepdim=True)

torch.save(spot_abs_ADCAA,  "ADCAA_spot_factors_abs.pt")
torch.save(spot_rel_ADCAA,  "ADCAA_spot_factors_rel.pt")
torch.save(spot_abs_CTRL,   "CTRL_spot_factors_abs.pt")
torch.save(spot_rel_CTRL,   "CTRL_spot_factors_rel.pt")

In [13]:
#When to Use each abundance:
#Spatial Maps                 -> Absolute
#Disease vs Control           -> Relative 
#Stats (Wilcoxon/LMM)         -> Relative
#Cell Density interpretation  -> Absolute

In [14]:
# --- Extract gene-level factor signatures ---
comb2fact = ADCAA_means["AutoNormal.comb2fact"]        # (n_comb × n_fact)
gene_level = ADCAA_means["AutoNormal.gene_level"]      # (1 × n_genes)

# Effective factor-by-gene loadings
# shape: (n_fact × n_genes)
factor_gene_expr = (
    comb2fact.T @ torch.ones((comb2fact.shape[0], gene_level.shape[1]))
) * gene_level

factor_gene_expr = factor_gene_expr.numpy()

factor_gene_df = pd.DataFrame(
    factor_gene_expr,
    index=[f"Factor_{i+1}" for i in range(factor_gene_expr.shape[0])],
    columns=X_ref_ADCAA_sub.index
)

In [ ]:
#Since the original Seurat object annotation was performed on the combined data set
#Cell-type identity should be condition-independent 
#Abundance should be condition dependent 

#Cell-types are fixed
#Gene programs are context-modulated
#Spatial abundance will be condition modulated

#factors that come from the regression should be robust across conditions-> not defined by one 


In [15]:
# --- Control factor gene signatures ---
comb2fact_CTRL = CTRL_means["AutoNormal.comb2fact"]
gene_level_CTRL = CTRL_means["AutoNormal.gene_level"]

factor_gene_expr_CTRL = (
    comb2fact_CTRL.T @ torch.ones((comb2fact_CTRL.shape[0], gene_level_CTRL.shape[1]))
) * gene_level_CTRL

factor_gene_df_CTRL = pd.DataFrame(
    factor_gene_expr_CTRL.numpy(),
    index=[f"Factor_{i+1}" for i in range(factor_gene_expr_CTRL.shape[0])],
    columns=X_ref_Control_sub.index
)


In [25]:
#------ Extract Factor gene matrix-------
celltypes = list(X_ref_ADCAA_sub.columns)   # length 46

spot_factors_ADCAA = ADCAA_means["AutoNormal.spot_factors"]  # (145 × 46)

spot_abs_ADCAA_df = pd.DataFrame(
    spot_factors_ADCAA.numpy(),
    index=adata_wta_ADCAA_sub.obs_names,
    columns=celltypes
)

spot_rel_ADCAA_df = spot_abs_ADCAA_df.div(spot_abs_ADCAA_df.sum(axis=1), axis=0)

# save
spot_abs_ADCAA_df.to_csv("ADCAA_spot_abundance_abs.csv")
spot_rel_ADCAA_df.to_csv("ADCAA_spot_abundance_rel.csv")

In [26]:
celltypes = list(X_ref_ADCAA_sub.columns)

# Make sure Control signatures are in same order
X_ref_Control_sub = X_ref_Control_sub[celltypes]

spot_factors_CTRL = CTRL_means["AutoNormal.spot_factors"]  # (45 × 46)

spot_abs_CTRL_df = pd.DataFrame(
    spot_factors_CTRL.numpy(),
    index=adata_wta_Control_sub.obs_names,
    columns=celltypes
)

spot_rel_CTRL_df = spot_abs_CTRL_df.div(spot_abs_CTRL_df.sum(axis=1), axis=0)

spot_abs_CTRL_df.to_csv("CTRL_spot_abundance_abs.csv")
spot_rel_CTRL_df.to_csv("CTRL_spot_abundance_rel.csv")


In [27]:
assert spot_abs_ADCAA_df.shape[1] == len(celltypes) == 46
assert spot_abs_CTRL_df.shape[1]  == len(celltypes) == 46

# make sure the ordering is identical
assert (spot_abs_ADCAA_df.columns == spot_abs_CTRL_df.columns).all()


In [28]:
plot_ADCAA = pd.concat([adata_wta_ADCAA_sub.obs, spot_rel_ADCAA_df], axis=1)
plot_CTRL  = pd.concat([adata_wta_Control_sub.obs, spot_rel_CTRL_df], axis=1)

plot_ADCAA.to_csv("ADCAA_plot_table.csv")
plot_CTRL.to_csv("CTRL_plot_table.csv")


In [30]:
#CONSENSUSSSSSSSSSS
print(list(X_ref_ADCAA_sub.columns)[:46])
print(list(X_ref_Control_sub.columns)[:46])

['Oligodendrocytes4', 'Oligodendrocytes1', 'OPC1', 'InhNeuron1', 'InhNeuron2', 'ExNeuron9', 'SMC', 'Oligodendrocytes6', 'Astrocytes1', 'ExNeuron3', 'ExNeuron2', 'Endothelial', 'Astrocytes5', 'ExNeuron12', 'ExNeuron7', 'Astrocytes3', 'ExNeuron6', 'Microglia1', 'OPC2', 'InhNeuron3', 'Microglia2', 'ExNeuron8', 'ExNeuron1', 'InhNeuron5', 'Microglia4', 'ExNeuron5', 'Microglia5', 'ExNeuron4', 'VLMC2', 'ExNeuron10', 'InhNeuron4', 'ExNeuron11', 'Oligodendrocytes2', 'Astrocytes4', 'Pericytes', 'ExNeuron14', 'Microglia3', 'VLMC1', 'Fibroblast', 'Oligodendrocytes8', 'Oligodendrocytes5', 'Oligodendrocytes3', 'ExNeuron13', 'OPC3', 'Oligodendrocytes7', 'Astrocytes2']
['Oligodendrocytes4', 'Oligodendrocytes1', 'OPC1', 'InhNeuron1', 'InhNeuron2', 'ExNeuron9', 'SMC', 'Oligodendrocytes6', 'Astrocytes1', 'ExNeuron3', 'ExNeuron2', 'Endothelial', 'Astrocytes5', 'ExNeuron12', 'ExNeuron7', 'Astrocytes3', 'ExNeuron6', 'Microglia1', 'OPC2', 'InhNeuron3', 'Microglia2', 'ExNeuron8', 'ExNeuron1', 'InhNeuron5', 'M

In [31]:
celltypes = list(X_ref_ADCAA_sub.columns)  # identical for CTRL too

# --- AD+CAA ---
spot_factors_ADCAA = ADCAA_means["AutoNormal.spot_factors"]  # (145 x 46)

spot_abs_ADCAA_df = pd.DataFrame(
    spot_factors_ADCAA.numpy(),
    index=adata_wta_ADCAA_sub.obs_names,
    columns=celltypes
)

spot_rel_ADCAA_df = spot_abs_ADCAA_df.div(spot_abs_ADCAA_df.sum(axis=1), axis=0)

spot_abs_ADCAA_df.to_csv("ADCAA_spot_abundance_abs.csv")
spot_rel_ADCAA_df.to_csv("ADCAA_spot_abundance_rel.csv")

# --- Control ---
spot_factors_CTRL = CTRL_means["AutoNormal.spot_factors"]  # (45 x 46)

spot_abs_CTRL_df = pd.DataFrame(
    spot_factors_CTRL.numpy(),
    index=adata_wta_Control_sub.obs_names,
    columns=celltypes
)

spot_rel_CTRL_df = spot_abs_CTRL_df.div(spot_abs_CTRL_df.sum(axis=1), axis=0)

spot_abs_CTRL_df.to_csv("CTRL_spot_abundance_abs.csv")
spot_rel_CTRL_df.to_csv("CTRL_spot_abundance_rel.csv")


In [32]:
plot_ADCAA = pd.concat([adata_wta_ADCAA_sub.obs, spot_abs_ADCAA_df.add_prefix("abs_"), spot_rel_ADCAA_df.add_prefix("rel_")], axis=1)
plot_CTRL  = pd.concat([adata_wta_Control_sub.obs,  spot_abs_CTRL_df.add_prefix("abs_"),  spot_rel_CTRL_df.add_prefix("rel_")], axis=1)

plot_ADCAA.to_csv("ADCAA_plot_table.csv")
plot_CTRL.to_csv("CTRL_plot_table.csv")


In [33]:
adata_wta_ADCAA_sub.obsm["cell_abundance_abs"] = spot_factors_ADCAA.numpy()
adata_wta_ADCAA_sub.obsm["cell_abundance_rel"] = spot_rel_ADCAA_df.values

adata_wta_Control_sub.obsm["cell_abundance_abs"] = spot_factors_CTRL.numpy()
adata_wta_Control_sub.obsm["cell_abundance_rel"] = spot_rel_CTRL_df.values

adata_wta_ADCAA_sub.uns["celltypes_order"] = celltypes
adata_wta_Control_sub.uns["celltypes_order"] = celltypes

adata_wta_ADCAA_sub.write("ADCAA_with_abundances.h5ad")
adata_wta_Control_sub.write("CTRL_with_abundances.h5ad")


In [37]:
print(adata_wta.obsm.keys())
print(adata_wta.obs.columns)

KeysView(AxisArrays with keys: negProbes)
Index(['Scan Name', 'ROI', 'Segment (Name/ Label)', 'Segment Tags',
       'Custom Segment Name',
       'LOT_Human_NGS_Whole_Transcriptome_Atlas_RNA_1_0', 'ROI_ID', 'Scan_ID',
       'disease_status', 'pathology', 'region'],
      dtype='object')


In [38]:
celltypes = list(spot_abs_ADCAA_df.columns)

# AD-CAA
df_AD_abs = spot_abs_ADCAA_df.copy()
df_AD_rel = spot_rel_ADCAA_df.copy()
df_AD_abs["ROI"] = df_AD_abs.index
df_AD_rel["ROI"] = df_AD_rel.index

meta_AD = adata_wta_ADCAA_sub.obs.copy()
meta_AD["ROI"] = meta_AD.index

df_AD = meta_AD.merge(df_AD_abs, on="ROI").merge(
    df_AD_rel.add_prefix("rel_"), left_on="ROI", right_on="rel_ROI"
).drop(columns=["rel_ROI"])

# Control
df_C_abs = spot_abs_CTRL_df.copy()
df_C_rel = spot_rel_CTRL_df.copy()
df_C_abs["ROI"] = df_C_abs.index
df_C_rel["ROI"] = df_C_rel.index

meta_C = adata_wta_Control_sub.obs.copy()
meta_C["ROI"] = meta_C.index

df_C = meta_C.merge(df_C_abs, on="ROI").merge(
    df_C_rel.add_prefix("rel_"), left_on="ROI", right_on="rel_ROI"
).drop(columns=["rel_ROI"])

# Combine
df_all = pd.concat([df_AD, df_C], axis=0, ignore_index=True)

# Save a wide plot-ready table
df_all.to_csv("cell2location_abundance_wide_with_meta.csv", index=False)

print(df_all.shape)
print(df_all[["disease_status","pathology","region"]].value_counts())

(190, 103)
disease_status  pathology    region     
AD-CAA          Amyloid      Parenchyma     32
                AmyloidFree  Arteries       29
                             Parenchyma     28
                Amyloid      Arteries       26
                AmyloidFree  Capillaries    17
Control         AmyloidFree  Arteries       17
                             Parenchyma     16
AD-CAA          Amyloid      Capillaries    13
Control         AmyloidFree  Capillaries    12
Name: count, dtype: int64


In [40]:
# Melt RELATIVE abundances into long format
rel_cols = [f"rel_{ct}" for ct in celltypes]

df_long = df_all.melt(
    id_vars=["ROI", "Scan_ID", "disease_status", "pathology", "region"],
    value_vars=rel_cols,
    var_name="celltype",
    value_name="rel_abundance"
)

df_long["celltype"] = df_long["celltype"].str.replace("^rel_", "", regex=True)

df_long.to_csv("cell2location_abundance_long_rel.csv", index=False)
df_long.head()


,ROI,Scan_ID,disease_status,pathology,region,celltype,rel_abundance
0,0,BRC 2534 BRC 2732,AD-CAA,Amyloid,Arteries,Oligodendrocytes4,0.027669
1,1,BRC 2534 BRC 2732,AD-CAA,Amyloid,Arteries,Oligodendrocytes4,0.021056
2,2,BRC 2534 BRC 2732,AD-CAA,Amyloid,Arteries,Oligodendrocytes4,0.021896
3,3,BRC 2534 BRC 2732,AD-CAA,Amyloid,Arteries,Oligodendrocytes4,0.020941
4,4,BRC 2534 BRC 2732,AD-CAA,Amyloid,Arteries,Oligodendrocytes4,0.033452


In [41]:
import sys
for module in sys.modules:
    try:
        print(module,sys.modules[module].__version__)
    except:
        try:
            if  type(sys.modules[module].version) is str:
                print(module,sys.modules[module].version)
            else:
                print(module,sys.modules[module].version())
        except:
            try:
                print(module,sys.modules[module].VERSION)
            except:
                pass

sys 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:29:10) [GCC 14.3.0]
re 2.2.1
ipaddress 1.0
ipykernel._version 7.1.0
json 2.0.9
jupyter_client._version 8.6.3
platform 1.0.8
zmq.sugar.version 27.1.0
zmq.sugar 27.1.0
zmq 27.1.0
logging 0.5.1.2
traitlets._version 5.14.3
traitlets 5.14.3
jupyter_core.version 5.9.1
jupyter_core 5.9.1
tornado 6.5.2
zlib 1.0
_ctypes 1.1.0
ctypes 1.1.0
colorama 0.4.6
_curses b'2.2'
socketserver 0.4
argparse 1.1
dateutil._version 2.9.0.post0
dateutil 2.9.0.post0
six 1.17.0
_decimal 1.70
decimal 1.70
platformdirs.version 4.5.0
platformdirs 4.5.0
_csv 1.0
csv 1.0
jupyter_client 8.6.3
ipykernel 7.1.0
IPython.core.release 8.37.0
executing.version 2.2.1
executing 2.2.1
pure_eval.version 0.2.3
pure_eval 0.2.3
stack_data.version 0.6.3
stack_data 0.6.3
pygments 2.19.2
IPython.core.crashhandler 8.37.0
pickleshare 0.7.5
decorator 5.2.1
_sqlite3 2.6.0
sqlite3.dbapi2 2.6.0
sqlite3 2.6.0
exceptiongroup._version 1.3.0
exceptiongroup 1.3.0
wcwidth 0.2.14
prompt_

/tmp/ipykernel_1073236/1799866556.py:4: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.__version__.
  print(module,sys.modules[module].__version__)
/tmp/ipykernel_1073236/1799866556.py:4: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(module,sys.modules[module].__version__)
